# WiSig Module-2: 双分数诊断（类一致性 + 域一致性）— 域分数多方案对比 + TX 随机划分

本 notebook 做三件事：

1) 训练 **已知设备闭集分类器**（ResNet18-1D）得到 embedding `z`，输出 **类一致性分数** `S_id`（默认 prototype cosine）。  
2) 计算多种 **域一致性分数** `S_dom`（重点包含：**per-device × per-sourceRX mixture**）。  
3) 在固定协议下评估：Unknown-TX（类不一致）与 Known-Drift（域不一致：cross-RX / cross-day）的 AUC，并对比不同 `S_dom` 方案。

附加：每次运行会按 `SEED` **随机选择 known/unknown TX**（避免挑选偏置且可复现）。可设置 `TX_SPLIT_REPEATS>1` 做多次随机划分取均值。


In [13]:
import os, json, time
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---- WiSig loader (user-provided) ----
dataset_name = "ManySig"
dataset_path = "../ManySig.pkl/"

def load_compact_pkl_dataset(dataset_path,dataset_name):
    with open(dataset_path+dataset_name+'.pkl','rb') as f:
        dataset = pickle.load(f)
    return dataset

# from <your_loader_module> import load_compact_pkl_dataset
compact_dataset = load_compact_pkl_dataset(dataset_path, dataset_name)

# ---- subset sizes ----
TX_TOTAL_USE = 6
RX_TOTAL_USE = 12
DAY_TOTAL_USE = 4

KNOWN_TX_NUM = 4          # known TX count (rest are unknown)
SOURCE_RX_NUM = 9         # number of source RXs (random by SEED)
TX_SPLIT_REPEATS = 3      # repeat random known/unknown TX selection; set 1 for single split

# ---- domain protocol ----
TRAIN_DATES = ["2021_03_15"]        # source day(s)
TEST_DATES_RX = ["2021_03_15"]      # cross-RX drift on same day
TEST_DATES_DAY = ["2021_03_01"]     # cross-day drift on source RXs

# ---- which signal version to use ----
Z_FROM_EQ = 1            # classifier input: 1=equalized, 0=unequalized
D_FROM = "raw"           # domain descriptor source: "raw" or "eq"
MAX_SIG_PER_TRIPLE = None

# ---- training hyperparams ----
BATCH_SIZE = 128
EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 8
IN_PLANES = 64
DROPOUT = 0.3

# ---- feature dims ----
D_DIM = 32
TOPK_CLASS_FOR_DOM = 3

# ---- output dir ----
ts = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"./results_wisig_dom_variants/run_{ts}"
os.makedirs(RUN_DIR, exist_ok=True)

cfg = dict(
    SEED=SEED, dataset_name=dataset_name, dataset_path=dataset_path,
    TX_TOTAL_USE=TX_TOTAL_USE, RX_TOTAL_USE=RX_TOTAL_USE, DAY_TOTAL_USE=DAY_TOTAL_USE,
    KNOWN_TX_NUM=KNOWN_TX_NUM, SOURCE_RX_NUM=SOURCE_RX_NUM, TX_SPLIT_REPEATS=TX_SPLIT_REPEATS,
    TRAIN_DATES=TRAIN_DATES, TEST_DATES_RX=TEST_DATES_RX, TEST_DATES_DAY=TEST_DATES_DAY,
    Z_FROM_EQ=Z_FROM_EQ, D_FROM=D_FROM, MAX_SIG_PER_TRIPLE=MAX_SIG_PER_TRIPLE,
    BATCH_SIZE=BATCH_SIZE, EPOCHS=EPOCHS, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY, PATIENCE=PATIENCE,
    IN_PLANES=IN_PLANES, DROPOUT=DROPOUT,
    D_DIM=D_DIM, TOPK_CLASS_FOR_DOM=TOPK_CLASS_FOR_DOM
)
with open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)
print("RUN_DIR:", RUN_DIR)

Device: cuda
RUN_DIR: ./results_wisig_dom_variants/run_20260304_152039


In [14]:
# =========================
# Data helpers
# =========================
def get_idx(lst, val):
    return lst.index(val)

def subsample(sigs, max_sig):
    if max_sig is None:
        return sigs
    return sigs[:min(len(sigs), max_sig)]

def get_signals(compact_dataset, tx, rx, date, equalized):
    tx_i = get_idx(compact_dataset["tx_list"], tx)
    rx_i = get_idx(compact_dataset["rx_list"], rx)
    date_i = get_idx(compact_dataset["capture_date_list"], date)
    eq_i = get_idx(compact_dataset["equalized_list"], equalized)
    sigs = compact_dataset["data"][tx_i][rx_i][date_i][eq_i]
    return np.array(sigs, dtype=np.float32)

def iq_to_complex(x_256_2):
    return (x_256_2[...,0] + 1j*x_256_2[...,1]).astype(np.complex64)

def domain_feat_256_complex(x256_c, d_dim=32):
    x = x256_c / (np.sqrt(np.mean(np.abs(x256_c)**2)) + 1e-12)
    X = np.fft.fft(x, n=256)
    mag = np.abs(X) + 1e-12
    logm = np.log(mag)
    D = np.fft.rfft(logm, n=512)
    return np.real(D[:d_dim]).astype(np.float32)

def compute_domain_feats_batch(X_256_2, d_dim=32):
    Xc = iq_to_complex(X_256_2)
    return np.stack([domain_feat_256_complex(Xc[i], d_dim=d_dim) for i in range(Xc.shape[0])], axis=0).astype(np.float32)

In [15]:
# =========================
# Choose subset + random SOURCE RXs (fixed by SEED)
# =========================
tx_list = compact_dataset["tx_list"]
rx_list = compact_dataset["rx_list"]
date_list = compact_dataset["capture_date_list"]

TX_USE = tx_list[:TX_TOTAL_USE]
RX_USE = rx_list[:RX_TOTAL_USE]
DATE_USE = date_list[:DAY_TOTAL_USE]

rng_rx = np.random.RandomState(SEED)
SOURCE_RXS = list(rng_rx.choice(RX_USE, size=SOURCE_RX_NUM, replace=False))
SOURCE_RXS.sort()
DRIFT_RX = [r for r in RX_USE if r not in SOURCE_RXS]

print("TX_USE:", TX_USE)
print("RX_USE:", RX_USE)
print("DATE_USE:", DATE_USE)
print("SOURCE_RXS:", SOURCE_RXS)
print("DRIFT_RX:", DRIFT_RX)

TX_USE: ['14-10', '14-7', '20-15', '20-19', '6-15', '8-20']
RX_USE: ['1-1', '1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '7-14', '7-7', '8-8']
DATE_USE: ['2021_03_01', '2021_03_08', '2021_03_15', '2021_03_23']
SOURCE_RXS: ['1-1', '1-19', '14-7', '19-2', '2-1', '3-19', '7-14', '7-7', '8-8']
DRIFT_RX: ['18-2', '2-19', '20-1']


In [16]:
# =========================
# Model: ResNet18-1D
# =========================
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(p=dropout)
        self.conv2 = nn.Conv1d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        out = out + identity
        return self.relu(out)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes, in_planes=64, dropout=0.0):
        super().__init__()
        self.in_planes = in_planes
        self.conv1 = nn.Conv1d(2, in_planes, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(in_planes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(128, 2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(256, 2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(512, 2, stride=2, dropout=dropout)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, planes, blocks, stride, dropout):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(planes)
            )
        layers = [BasicBlock1D(self.in_planes, planes, stride, downsample, dropout)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.in_planes, planes, dropout=dropout))
        return nn.Sequential(*layers)

    def forward(self, x, return_embed=False):
        x = x.permute(0, 2, 1)  # (B,256,2)->(B,2,256)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)  # (B,512)
        logits = self.fc(x)
        if return_embed:
            return logits, x
        return logits

In [17]:
# =========================
# Train / embedding / scoring utils
# =========================
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return self.X[i], self.y[i]

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    crit = nn.CrossEntropyLoss()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * yb.size(0)
        pred = logits.argmax(dim=1)
        total_correct += int((pred == yb).sum().item())
        total += int(yb.size(0))
    return total_loss / max(1,total), total_correct / max(1,total)

def compute_embeddings(model, X_np, batch=512):
    model.eval()
    ds = ArrayDataset(X_np, np.zeros((X_np.shape[0],), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    Z = []
    with torch.no_grad():
        for xb, _ in dl:
            xb = xb.to(device)
            _, emb = model(xb, return_embed=True)
            Z.append(emb.cpu().numpy().astype(np.float32))
    return np.concatenate(Z, axis=0)

def l2norm(x, axis=1, eps=1e-12):
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + eps)

def prototypes(Z, y, K):
    ZN = l2norm(Z, axis=1)
    P = np.zeros((K, Z.shape[1]), dtype=np.float32)
    for k in range(K):
        idx = np.where(y==k)[0]
        P[k] = ZN[idx].mean(axis=0)
    return l2norm(P, axis=1)

def cosine_to_proto(Z, P):
    ZN = l2norm(Z, axis=1)
    return ZN @ P.T

def mahalanobis_batch(D, mu, Sinv):
    X = D - mu.reshape(1,-1)
    return np.einsum("nd,dd,nd->n", X, Sinv, X).astype(np.float32)

def fit_gaussian(D, reg=1e-3):
    mu = D.mean(axis=0).astype(np.float32)
    C = np.cov(D.T, bias=False).astype(np.float32)
    C = C + reg*np.eye(C.shape[0], dtype=np.float32)
    Sinv = np.linalg.inv(C).astype(np.float32)
    return mu, Sinv

def roc_auc(y_true, score):
    fpr, tpr, _ = roc_curve(y_true, score)
    return float(auc(fpr, tpr))

In [18]:
# =========================
# Build datasets (single-sample) under a given known/unknown TX split
# =========================
def build_source_train(compact_dataset, KNOWN_TX):
    X_list, y_list, D_list = [], [], []
    rx_id_list, day_id_list = [], []
    for tx in KNOWN_TX:
        for rx in SOURCE_RXS:
            for day in TRAIN_DATES:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                if D_FROM == "raw":
                    Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0), MAX_SIG_PER_TRIPLE)
                else:
                    Xd = subsample(get_signals(compact_dataset, tx, rx, day, 1), MAX_SIG_PER_TRIPLE)

                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]

                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)

                X_list.append(Xz); y_list.append(y); D_list.append(Df)
                rx_id_list.append(np.full((n,), RX_USE.index(rx), dtype=np.int64))
                day_id_list.append(np.full((n,), TRAIN_DATES.index(day), dtype=np.int64))

    X = np.concatenate(X_list, axis=0).astype(np.float32)
    y = np.concatenate(y_list, axis=0).astype(np.int64)
    D = np.concatenate(D_list, axis=0).astype(np.float32)
    rx_id = np.concatenate(rx_id_list, axis=0).astype(np.int64)
    day_id = np.concatenate(day_id_list, axis=0).astype(np.int64)
    return X, y, D, rx_id, day_id

def build_eval_set(compact_dataset, KNOWN_TX, txs, rxs, days):
    X_list, y_list, D_list = [], [], []
    for tx in txs:
        for rx in rxs:
            for day in days:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                if D_FROM == "raw":
                    Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0), MAX_SIG_PER_TRIPLE)
                else:
                    Xd = subsample(get_signals(compact_dataset, tx, rx, day, 1), MAX_SIG_PER_TRIPLE)

                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]

                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64) if tx in KNOWN_TX else np.full((n,), -1, dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)

                X_list.append(Xz); y_list.append(y); D_list.append(Df)

    X = np.concatenate(X_list, axis=0).astype(np.float32)
    y = np.concatenate(y_list, axis=0).astype(np.int64)
    D = np.concatenate(D_list, axis=0).astype(np.float32)
    return X, y, D

In [19]:
# =========================
# Domain score variants
# =========================
def domfit_V0_per_device(D_train, y_train, K):
    models = []
    for k in range(K):
        models.append(fit_gaussian(D_train[y_train==k]))
    return models

def domscore_V0(D_eval, k_hat, models):
    out = np.zeros((D_eval.shape[0],), dtype=np.float32)
    for k,(mu,Sinv) in enumerate(models):
        idx = np.where(k_hat==k)[0]
        if idx.size:
            out[idx] = mahalanobis_batch(D_eval[idx], mu, Sinv)
    return out

def domfit_V1_device_rx(D_train, y_train, rx_train, K, source_rx_ids):
    models_kr = {}
    for k in range(K):
        for rxid in source_rx_ids:
            idx = np.where((y_train==k) & (rx_train==rxid))[0]
            if idx.size >= 10:
                models_kr[(k, rxid)] = fit_gaussian(D_train[idx])
    fallback_k = domfit_V0_per_device(D_train, y_train, K)
    return models_kr, fallback_k

def domscore_V1(D_eval, k_hat, models_kr, fallback_k, source_rx_ids):
    out = np.zeros((D_eval.shape[0],), dtype=np.float32)
    for i in range(D_eval.shape[0]):
        k = int(k_hat[i])
        d = D_eval[i:i+1]
        vals = []
        for rxid in source_rx_ids:
            if (k, rxid) in models_kr:
                mu, Sinv = models_kr[(k, rxid)]
            else:
                mu, Sinv = fallback_k[k]
            vals.append(float(mahalanobis_batch(d, mu, Sinv)[0]))
        out[i] = np.min(vals)
    return out

def domscore_V1_topK(D_eval, cos_scores, models_kr, fallback_k, source_rx_ids, topk=3):
    N, K = cos_scores.shape
    topk = max(1, min(topk, K))
    top_idx = np.argpartition(-cos_scores, kth=topk-1, axis=1)[:, :topk]
    out = np.zeros((N,), dtype=np.float32)
    for i in range(N):
        d = D_eval[i:i+1]
        best = 1e9
        for k in top_idx[i]:
            kk = int(k)
            for rxid in source_rx_ids:
                if (kk, rxid) in models_kr:
                    mu, Sinv = models_kr[(kk, rxid)]
                else:
                    mu, Sinv = fallback_k[kk]
                val = float(mahalanobis_batch(d, mu, Sinv)[0])
                if val < best:
                    best = val
        out[i] = best
    return out

def domfit_V2_per_rx(D_train, rx_train, source_rx_ids):
    models = []
    for rxid in source_rx_ids:
        models.append(fit_gaussian(D_train[rx_train==rxid]))
    return models

def domfit_V3_per_day(D_train, day_train, source_day_ids):
    models = []
    for did in source_day_ids:
        models.append(fit_gaussian(D_train[day_train==did]))
    return models

def domscore_min_models(D_eval, models):
    dists = [mahalanobis_batch(D_eval, mu, Sinv) for (mu,Sinv) in models]
    return np.min(np.stack(dists, axis=1), axis=1).astype(np.float32)

In [20]:
# =========================
# Main experiment loop: repeats over random TX splits; 5-fold CV on source samples
# =========================
def run_one_split(split_id, KNOWN_TX, UNKNOWN_TX):
    split_dir = os.path.join(RUN_DIR, f"txsplit_{split_id}")
    os.makedirs(split_dir, exist_ok=True)
    with open(os.path.join(split_dir, "tx_split.json"), "w", encoding="utf-8") as f:
        json.dump({"KNOWN_TX": KNOWN_TX, "UNKNOWN_TX": UNKNOWN_TX, "SOURCE_RXS": SOURCE_RXS, "DRIFT_RX": DRIFT_RX}, f, indent=2)

    X_src, y_src, D_src, rx_src, day_src = build_source_train(compact_dataset, KNOWN_TX)
    K = len(KNOWN_TX)

    # eval sets: never used for training
    X_drRX, y_drRX, D_drRX = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, DRIFT_RX, TEST_DATES_RX)
    X_drDAY, y_drDAY, D_drDAY = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, SOURCE_RXS, TEST_DATES_DAY)
    X_u, y_u, D_u = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, SOURCE_RXS, TEST_DATES_RX)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1000*split_id)

    rows = []
    for fold, (idx_tr, idx_te) in enumerate(skf.split(X_src, y_src), start=1):
        fold_dir = os.path.join(split_dir, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        # val split
        rng = np.random.RandomState(SEED + 1000*split_id + fold)
        perm = rng.permutation(idx_tr)
        n_val = max(1, int(0.1 * len(perm)))
        idx_val = perm[:n_val]
        idx_train = perm[n_val:]

        train_loader = DataLoader(ArrayDataset(X_src[idx_train], y_src[idx_train]), batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(ArrayDataset(X_src[idx_val],   y_src[idx_val]),   batch_size=BATCH_SIZE, shuffle=False)
        test_loader  = DataLoader(ArrayDataset(X_src[idx_te],    y_src[idx_te]),    batch_size=BATCH_SIZE, shuffle=False)

        model = ResNet18_1D(num_classes=K, in_planes=IN_PLANES, dropout=DROPOUT).to(device)
        opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        best_val = -1.0
        best_state = None
        patience = 0
        hist = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[]}

        for ep in range(1, EPOCHS+1):
            tr_loss, tr_acc = run_epoch(model, train_loader, optimizer=opt)
            va_loss, va_acc = run_epoch(model, val_loader, optimizer=None)
            hist["train_loss"].append(tr_loss); hist["train_acc"].append(tr_acc)
            hist["val_loss"].append(va_loss);   hist["val_acc"].append(va_acc)
            if va_acc > best_val + 1e-4:
                best_val = va_acc
                best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= PATIENCE:
                    break

        torch.save(best_state, os.path.join(fold_dir, "best_model.pth"))
        with open(os.path.join(fold_dir, "history.json"), "w", encoding="utf-8") as f:
            json.dump(hist, f, indent=2)

        model.load_state_dict(best_state)

        # in-domain acc on held-out source
        model.eval()
        y_pred = []
        with torch.no_grad():
            for xb, _yb in test_loader:
                xb = xb.to(device)
                logits = model(xb)
                y_pred.append(logits.argmax(dim=1).cpu().numpy())
        y_pred = np.concatenate(y_pred)
        acc = float((y_pred == y_src[idx_te]).mean())

        # prototypes for S_id and class scores
        Z_train = compute_embeddings(model, X_src[idx_train], batch=512)
        P = prototypes(Z_train, y_src[idx_train], K)

        def sid_cos_khat(Z):
            cos = cosine_to_proto(Z, P)
            Sid = np.max(cos, axis=1).astype(np.float32)
            k1 = np.argmax(cos, axis=1).astype(np.int64)
            return Sid, cos, k1

        # stable A
        Z_A = compute_embeddings(model, X_src[idx_te], batch=512)
        Sid_A, cos_A, kA = sid_cos_khat(Z_A)
        D_A = D_src[idx_te]

        # drift RX B
        Z_B = compute_embeddings(model, X_drRX, batch=512)
        Sid_B, cos_B, kB = sid_cos_khat(Z_B)

        # drift DAY E
        Z_E = compute_embeddings(model, X_drDAY, batch=512)
        Sid_E, cos_E, kE = sid_cos_khat(Z_E)

        # unknown U
        Z_U = compute_embeddings(model, X_u, batch=512)
        Sid_U, cos_U, kU = sid_cos_khat(Z_U)

        # Unknown AUC
        y_unknown = np.concatenate([np.zeros_like(Sid_A, dtype=np.int64), np.ones_like(Sid_U, dtype=np.int64)])
        s_unknown = np.concatenate([-Sid_A, -Sid_U])
        auc_u = roc_auc(y_unknown, s_unknown)

        # domain model fit on TRAIN only
        source_rx_ids = sorted({RX_USE.index(r) for r in SOURCE_RXS})
        source_day_ids = sorted(set(day_src[idx_train].tolist()))

        V0 = domfit_V0_per_device(D_src[idx_train], y_src[idx_train], K)
        V1_kr, V1_fb = domfit_V1_device_rx(D_src[idx_train], y_src[idx_train], rx_src[idx_train], K, source_rx_ids)
        V2 = domfit_V2_per_rx(D_src[idx_train], rx_src[idx_train], source_rx_ids)
        V3 = domfit_V3_per_day(D_src[idx_train], day_src[idx_train], source_day_ids)

        # compute Sdom for variants
        SdomA_V0 = domscore_V0(D_A, kA, V0)
        SdomB_V0 = domscore_V0(D_drRX, kB, V0)
        SdomE_V0 = domscore_V0(D_drDAY, kE, V0)

        SdomA_V1  = domscore_V1(D_A, kA, V1_kr, V1_fb, source_rx_ids)
        SdomB_V1  = domscore_V1(D_drRX, kB, V1_kr, V1_fb, source_rx_ids)
        SdomE_V1  = domscore_V1(D_drDAY, kE, V1_kr, V1_fb, source_rx_ids)

        SdomA_V1k = domscore_V1_topK(D_A, cos_A, V1_kr, V1_fb, source_rx_ids, topk=TOPK_CLASS_FOR_DOM)
        SdomB_V1k = domscore_V1_topK(D_drRX, cos_B, V1_kr, V1_fb, source_rx_ids, topk=TOPK_CLASS_FOR_DOM)
        SdomE_V1k = domscore_V1_topK(D_drDAY, cos_E, V1_kr, V1_fb, source_rx_ids, topk=TOPK_CLASS_FOR_DOM)

        SdomA_V2  = domscore_min_models(D_A, V2)
        SdomB_V2  = domscore_min_models(D_drRX, V2)
        SdomE_V2  = domscore_min_models(D_drDAY, V2)

        SdomA_V3  = domscore_min_models(D_A, V3)
        SdomB_V3  = domscore_min_models(D_drRX, V3)
        SdomE_V3  = domscore_min_models(D_drDAY, V3)

        def auc_drift(a, b):
            y = np.concatenate([np.zeros_like(a, dtype=np.int64), np.ones_like(b, dtype=np.int64)])
            s = np.concatenate([a, b])
            return roc_auc(y, s)

        # RX drift
        auc_drRX_V0  = auc_drift(SdomA_V0,  SdomB_V0)
        auc_drRX_V1  = auc_drift(SdomA_V1,  SdomB_V1)
        auc_drRX_V1k = auc_drift(SdomA_V1k, SdomB_V1k)
        auc_drRX_V2  = auc_drift(SdomA_V2,  SdomB_V2)
        auc_drRX_V3  = auc_drift(SdomA_V3,  SdomB_V3)

        # DAY drift
        auc_drDAY_V0  = auc_drift(SdomA_V0,  SdomE_V0)
        auc_drDAY_V1  = auc_drift(SdomA_V1,  SdomE_V1)
        auc_drDAY_V1k = auc_drift(SdomA_V1k, SdomE_V1k)
        auc_drDAY_V2  = auc_drift(SdomA_V2,  SdomE_V2)
        auc_drDAY_V3  = auc_drift(SdomA_V3,  SdomE_V3)

        row = dict(
            split=split_id, fold=fold, acc=acc, auc_unknown=auc_u,
            auc_drRX_V0=auc_drRX_V0, auc_drRX_V1=auc_drRX_V1, auc_drRX_V1k=auc_drRX_V1k, auc_drRX_V2=auc_drRX_V2, auc_drRX_V3=auc_drRX_V3,
            auc_drDAY_V0=auc_drDAY_V0, auc_drDAY_V1=auc_drDAY_V1, auc_drDAY_V1k=auc_drDAY_V1k, auc_drDAY_V2=auc_drDAY_V2, auc_drDAY_V3=auc_drDAY_V3,
        )
        rows.append(row)
        with open(os.path.join(fold_dir, "metrics_variants.json"), "w", encoding="utf-8") as f:
            json.dump(row, f, indent=2)

        print(f"[split {split_id} fold {fold}] acc={acc:.3f} AUC_u={auc_u:.3f}  drRX(V1k)={auc_drRX_V1k:.3f} drDAY(V3)={auc_drDAY_V3:.3f}")

    with open(os.path.join(split_dir, "metrics_all_folds.json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    return rows

all_rows = []
for split_id in range(1, TX_SPLIT_REPEATS+1):
    rng_tx = np.random.RandomState(SEED + 777*split_id)
    tx_perm = list(rng_tx.permutation(TX_USE))
    KNOWN_TX = tx_perm[:KNOWN_TX_NUM]
    UNKNOWN_TX = tx_perm[KNOWN_TX_NUM:]
    print("\n=== TX split", split_id, "===")
    print("KNOWN_TX:", KNOWN_TX)
    print("UNKNOWN_TX:", UNKNOWN_TX)
    rows = run_one_split(split_id, KNOWN_TX, UNKNOWN_TX)
    all_rows.extend(rows)

with open(os.path.join(RUN_DIR, "metrics_all_splits_folds.json"), "w", encoding="utf-8") as f:
    json.dump(all_rows, f, indent=2)

print("\nSaved all metrics to:", RUN_DIR)


=== TX split 1 ===
KNOWN_TX: ['20-15', '20-19', '14-10', '8-20']
UNKNOWN_TX: ['6-15', '14-7']
[split 1 fold 1] acc=0.997 AUC_u=0.803  drRX(V1k)=0.830 drDAY(V3)=0.503
[split 1 fold 2] acc=0.995 AUC_u=0.698  drRX(V1k)=0.815 drDAY(V3)=0.504
[split 1 fold 3] acc=0.998 AUC_u=0.819  drRX(V1k)=0.829 drDAY(V3)=0.509
[split 1 fold 4] acc=0.998 AUC_u=0.903  drRX(V1k)=0.815 drDAY(V3)=0.510
[split 1 fold 5] acc=0.998 AUC_u=0.807  drRX(V1k)=0.838 drDAY(V3)=0.506

=== TX split 2 ===
KNOWN_TX: ['8-20', '6-15', '20-15', '14-10']
UNKNOWN_TX: ['20-19', '14-7']
[split 2 fold 1] acc=0.997 AUC_u=0.822  drRX(V1k)=0.870 drDAY(V3)=0.519
[split 2 fold 2] acc=0.998 AUC_u=0.894  drRX(V1k)=0.872 drDAY(V3)=0.517
[split 2 fold 3] acc=0.998 AUC_u=0.853  drRX(V1k)=0.855 drDAY(V3)=0.514
[split 2 fold 4] acc=0.998 AUC_u=0.933  drRX(V1k)=0.842 drDAY(V3)=0.517
[split 2 fold 5] acc=0.997 AUC_u=0.942  drRX(V1k)=0.878 drDAY(V3)=0.517

=== TX split 3 ===
KNOWN_TX: ['14-7', '20-19', '8-20', '6-15']
UNKNOWN_TX: ['20-15', '14-

In [21]:
# =========================
# Aggregate summary (mean±std) over splits×folds
# =========================
import numpy as np

def mean_std(arr):
    arr = np.array(arr, dtype=np.float64)
    return float(arr.mean()), float(arr.std(ddof=0))

keys = [
    "acc","auc_unknown",
    "auc_drRX_V0","auc_drRX_V1","auc_drRX_V1k","auc_drRX_V2","auc_drRX_V3",
    "auc_drDAY_V0","auc_drDAY_V1","auc_drDAY_V1k","auc_drDAY_V2","auc_drDAY_V3"
]

summary = {}
for k in keys:
    m,s = mean_std([r[k] for r in all_rows])
    summary[k] = {"mean": m, "std": s}

with open(os.path.join(RUN_DIR, "summary_mean_std.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("=== Mean ± Std over splits×folds ===")
for k in keys:
    m,s = summary[k]["mean"], summary[k]["std"]
    print(f"{k:14s}: {m:.3f} ± {s:.3f}")

=== Mean ± Std over splits×folds ===
acc           : 0.997 ± 0.001
auc_unknown   : 0.874 ± 0.070
auc_drRX_V0   : 0.706 ± 0.019
auc_drRX_V1   : 0.866 ± 0.018
auc_drRX_V1k  : 0.833 ± 0.025
auc_drRX_V2   : 0.527 ± 0.061
auc_drRX_V3   : 0.641 ± 0.029
auc_drDAY_V0  : 0.530 ± 0.010
auc_drDAY_V1  : 0.736 ± 0.014
auc_drDAY_V1k : 0.721 ± 0.013
auc_drDAY_V2  : 0.519 ± 0.023
auc_drDAY_V3  : 0.499 ± 0.018
